<h2>Middleware</h2>

provides a way to more tightly control the agents behaviour.

built in middleware:
1. summarization middleware
2. human in the loop 
3. model call limit
4. tool call limit and much more


In [2]:
import os 
from dotenv import load_dotenv
load_dotenv()

os.environ["GROQ_API_KEY"]=os.getenv("GROQ_API_KEY")

In [6]:
# summarization middleware

from langchain.agents import create_agent
from langchain.chat_models import init_chat_model
from langchain.agents.middleware import SummarizationMiddleware
from langgraph.checkpoint.memory import InMemorySaver
from langchain_core.messages import HumanMessage, SystemMessage

agent= create_agent(model=init_chat_model(model="llama-3.3-70b-versatile",model_provider="groq",temperature="0.7"),
                    
  checkpointer=InMemorySaver(),
  middleware=[
      SummarizationMiddleware(
          model=init_chat_model(model="llama-3.3-70b-versatile",model_provider="groq",temperature="0.7"),
          trigger=("messages",10),
          keep=("messages",4)
      )
  ]                  
                    )

In [7]:
from langchain.agents import create_agent
from langchain.chat_models import init_chat_model
from langchain.agents.middleware import SummarizationMiddleware
from langgraph.checkpoint.memory import InMemorySaver

# Summarization Middleware in LangChain Agents

# The SummarizationMiddleware helps manage conversation history by summarizing older messages.
# It can trigger summarization based on different criteria, such as number of messages or token count.

# 1. Summarization based on number of messages (as in the existing agent):
# - trigger=("messages", 10): Summarize when there are more than 10 messages.
# - keep=("messages", 4): Retain the last 4 messages after summarization.
# This keeps the conversation concise while preserving recent context.

# 2. Summarization based on token size:
# - trigger=("tokens", 1000): Summarize when the total token count exceeds 1000.
# - keep=("messages", 4): Retain the last 4 messages.
# Tokens are units of text (e.g., words or subwords) used by the model; this prevents hitting token limits.

# Example of creating an agent with token-based summarization:

agent_token_based = create_agent(
    model=init_chat_model(model="llama-3.3-70b-versatile", model_provider="groq", temperature="0.7"),
    checkpointer=InMemorySaver(),
    middleware=[
        SummarizationMiddleware(
            model=init_chat_model(model="llama-3.3-70b-versatile", model_provider="groq", temperature="0.7"),
            trigger=("tokens", 1000),  # Trigger on token count
            keep=("messages", 4)
        )
    ]
)

# Usage: Invoke the agent with messages, and it will summarize based on tokens.
# result = agent_token_based.invoke({"messages": [HumanMessage(content="Your query here")]})

In [10]:
# Invoke the token-based summarization agent with a sample query
result = agent_token_based.invoke({"messages": [HumanMessage(content="What is the capital of France?")]}, config={"configurable": {"thread_id": "1"}})
print(result)

{'messages': [HumanMessage(content='What is the capital of France?', additional_kwargs={}, response_metadata={}, id='afd80723-66ff-4fd1-a319-416b4fd87755'), AIMessage(content='The capital of France is Paris.', additional_kwargs={}, response_metadata={'token_usage': {'completion_tokens': 8, 'prompt_tokens': 42, 'total_tokens': 50, 'completion_time': 0.011157844, 'completion_tokens_details': None, 'prompt_time': 0.002419594, 'prompt_tokens_details': None, 'queue_time': 0.049471776, 'total_time': 0.013577438}, 'model_name': 'llama-3.3-70b-versatile', 'system_fingerprint': 'fp_dae98b5ecb', 'service_tier': 'on_demand', 'finish_reason': 'stop', 'logprobs': None, 'model_provider': 'groq'}, id='lc_run--019d165f-f4ee-7d23-9ac5-32424ce550ed-0', tool_calls=[], invalid_tool_calls=[], usage_metadata={'input_tokens': 42, 'output_tokens': 8, 'total_tokens': 50})]}


In [9]:
# Check for errors in agent creation and invocation

try:
    # Test the original agent
    result = agent.invoke({"messages": [HumanMessage(content="Test query")]})
    print("Original agent works fine.")
except Exception as e:
    print(f"Error in original agent: {e}")

try:
    # Test the token-based agent
    result = agent_token_based.invoke({"messages": [HumanMessage(content="Test query")]})
    print("Token-based agent works fine.")
except Exception as e:
    print(f"Error in token-based agent: {e}")

Error in original agent: Checkpointer requires one or more of the following 'configurable' keys: thread_id, checkpoint_ns, checkpoint_id
Error in token-based agent: Checkpointer requires one or more of the following 'configurable' keys: thread_id, checkpoint_ns, checkpoint_id


In [11]:
# Example of messages-based summarization
# This demonstrates how the middleware summarizes when message count exceeds threshold

messages = [
    HumanMessage(content="Hello, how are you?"),
    SystemMessage(content="I'm doing well, thank you!"),
    HumanMessage(content="Can you tell me about Python?"),
    SystemMessage(content="Python is a programming language..."),
    HumanMessage(content="What about machine learning?"),
    SystemMessage(content="Machine learning is a subset of AI..."),
    HumanMessage(content="Tell me more about LangChain."),
    SystemMessage(content="LangChain is a framework for building LLM applications..."),
    HumanMessage(content="How does it work with Groq?"),
    SystemMessage(content="Groq provides fast inference for LLMs..."),
    HumanMessage(content="What are the benefits?"),
    SystemMessage(content="Benefits include speed, cost-effectiveness..."),
]

# Invoke with many messages to trigger summarization
result = agent.invoke({"messages": messages}, config={"configurable": {"thread_id": "2"}})
print("Result with message-based summarization:")
print(result)

Result with message-based summarization:
{'messages': [HumanMessage(content="Here is a summary of the conversation to date:\n\n## SESSION INTENT\nThe user's primary goal is to inquire about various topics related to programming and artificial intelligence, specifically Python, machine learning, and LangChain.\n\n## SUMMARY\nThe conversation started with a greeting, followed by the user's request for information about Python, machine learning, and LangChain. The system provided brief descriptions of each topic, indicating that Python is a programming language, machine learning is a subset of AI, and LangChain is a framework for building LLM applications.\n\n## ARTIFACTS\nNone\n\n## NEXT STEPS\nThe user may ask for more detailed information about Python, machine learning, or LangChain, or explore related topics such as AI applications or programming frameworks.", additional_kwargs={'lc_source': 'summarization'}, response_metadata={}, id='b0a79748-780b-445d-9a53-32129c398331'), HumanMessa

In [12]:
# Example of token-based summarization
# This demonstrates how the middleware summarizes when token count exceeds threshold

# Create a long message to exceed token limit
long_message = "What is artificial intelligence? " * 100  # Repeat to create long content

result_token = agent_token_based.invoke({"messages": [HumanMessage(content=long_message)]}, config={"configurable": {"thread_id": "3"}})
print("Result with token-based summarization:")
print(result_token)

Result with token-based summarization:
{'messages': [HumanMessage(content='What is artificial intelligence? What is artificial intelligence? What is artificial intelligence? What is artificial intelligence? What is artificial intelligence? What is artificial intelligence? What is artificial intelligence? What is artificial intelligence? What is artificial intelligence? What is artificial intelligence? What is artificial intelligence? What is artificial intelligence? What is artificial intelligence? What is artificial intelligence? What is artificial intelligence? What is artificial intelligence? What is artificial intelligence? What is artificial intelligence? What is artificial intelligence? What is artificial intelligence? What is artificial intelligence? What is artificial intelligence? What is artificial intelligence? What is artificial intelligence? What is artificial intelligence? What is artificial intelligence? What is artificial intelligence? What is artificial intelligence? W

In [13]:
# Summary of Middleware Examples
print("1. Messages-based Summarization: Summarizes when message count > 10, keeps last 4")
print("2. Token-based Summarization: Summarizes when token count exceeds threshold")
print("Both use LangGraph agents with checkpointers and middleware for conversation management.")

1. Messages-based Summarization: Summarizes when message count > 10, keeps last 4
2. Token-based Summarization: Summarizes when token count exceeds threshold
Both use LangGraph agents with checkpointers and middleware for conversation management.
